# Preprocessing for topic modelling

### Topic modelling preprocessing pipeline overview

Input: `stylecom_cleaned.csv`

Output: `checkpoints/df_with_noun_tokens.pkl`

Intermediate outputs: `fashion_ontology.csv`, `fashionpedia_ontology.csv`

- **1. Setup:** load dependencies, define paths and column names, build a brand/designer lookup from the dataset metadata
- **2. Fashion ontology:** construct a vocabulary of fashion-relevant terms from two sources:
    - a curated seed list of nouns and adjectives
    - the [Fashionpedia](https://fashionpedia.github.io/) dataset 
    - lemmatise all terms and index them into unigram, bigram, and trigram lookup sets
- **3. POS tagging & token filtering:** normalise review text (expand contractions, strip punctuation), run spaCy POS tagging, then filter down to content-bearing nouns and adjectives
    - excluding stopwords, brand names, and domain-generic words such as "collection"
- **4. Checkpoint:** save the dataframe with per-document noun token list for further processing

### Key variables
- `ontology_df`: term → POS category lookup, used to guide token filtering
- `token_df`: one row per token, with lemma, POS, and brand flags
- `df["noun_tokens"]`: list of filtered lemmas per review, used as input to the topic model

In [1]:
import re
import string
from pathlib import Path

import pandas as pd
import requests
import json
import spacy
import numpy as np
from scipy.stats import entropy as scipy_entropy
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use("Agg")

# 1. Preprocess data with SpaCy

In [2]:
# Cell 1: configuration

DATA_PATH     = Path("data/stylecom_cleaned.csv")
ONTOLOGY_PATH = Path("data/fashion_ontology.csv")
CHECKPOINT_DIR = Path("checkpoints")
OUT_DIR       = Path("outputs_rlda_preprocessing")

TEXT_COL      = "review"
BRAND_COL     = "designer"
DATE_COL      = "date"

In [3]:
# Cell 2: Load data from checkpoint or raw CSV

checkpoint_path = CHECKPOINT_DIR / "df_with_noun_tokens.pkl"

if checkpoint_path.exists():
    df = pd.read_pickle(checkpoint_path)
    print(f"Loaded from checkpoint: {df.shape}")
else:
    df = pd.read_csv(DATA_PATH).reset_index(names="doc_id")
    print(f"Loaded from CSV: {df.shape}")

Loaded from checkpoint: (6629, 15)


In [4]:
# Cell 3: Load raw data

print(df.shape)
print(df.columns.tolist())
df.head()

(6629, 15)
['doc_id', 'year', 'season', 'designer', 'author', 'city', 'date', 'review', '_date', 'review_norm', 'brand_mentions', 'noun_tokens_x', 'noun_text', 'noun_tokens_y', 'noun_tokens']


,doc_id,year,season,designer,author,city,date,review,_date,review_norm,brand_mentions,noun_tokens_x,noun_text,noun_tokens_y,noun_tokens
0,0,2000,Spring,Matt Nye,Armand Limnander,New York,17-Sep-99,Designer Matt Nye's sophomore show featured a ...,1999-09-17,Designer Matt Nye s sophomore show featured a ...,"[sophomore, matt nye]","[sophomore, coed, sailor, pant, jacket, simple...",sophomore coed sailor pant jacket simple cotto...,"[sophomore, coed, sailor, pant, jacket, simple...","[sophomore, coed, sailor, pant, jacket, simple..."
1,1,2000,Spring,Giorgio Armani,Armand Limnander,Milan,29-Sep-99,"Armani proposed a light, feminine silhouette f...",1999-09-29,Armani proposed a light feminine silhouette fo...,[sea],"[light, feminine, silhouette, millennium, line...",light feminine silhouette millennium line foam...,"[light, feminine, silhouette, millennium, line...","[light, feminine, silhouette, millennium, line..."
2,2,2000,Spring,Eric Bergère,Armand Limnander,Paris,4-Oct-99,Broadway Garnier was the theme for Eric Berg'r...,1999-10-04,Broadway Garnier was the theme for Eric Berg a...,[],"[tailleur, skirt, pleat, sweater, ruched, shir...",tailleur skirt pleat sweater ruched shirt inte...,"[tailleur, skirt, pleat, sweater, ruched, shir...","[tailleur, skirt, pleat, sweater, ruched, shir..."
3,3,2000,Spring,Céline,Armand Limnander,Paris,7-Oct-99,Getaway glamour was the theme for Celine's str...,1999-10-07,Getaway glamour was the theme for Celine s str...,"[michael kors, trademark]","[glamour, presentation, course, fun, sun, spea...",glamour presentation course fun sun speaker pr...,"[glamour, presentation, course, fun, sun, spea...","[glamour, presentation, course, fun, sun, spea..."
4,4,2000,Spring,Byblos,Armand Limnander,Milan,27-Sep-99,Judo Jetson blends my favorite cartoon charact...,1999-09-27,Judo Jetson blends my favorite cartoon charact...,"[john bartlett, byblos]","[favorite, cartoon, character, spiritual, japa...",favorite cartoon character spiritual japanese ...,"[favorite, cartoon, character, spiritual, japa...","[favorite, cartoon, character, spiritual, japa..."


In [5]:
# Cell 4: Build brand lookup from metadata

brand_names_from_metadata = (
    df[BRAND_COL]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

BRAND_NAMES  = sorted(set(brand_names_from_metadata))
BRAND_PHRASES = {b.lower() for b in BRAND_NAMES}
BRAND_WORDS   = {
    token.lower()
    for brand in BRAND_NAMES
    for token in re.findall(r"\b\w+\b", str(brand))
}

print(f"{len(BRAND_NAMES)} brand/designer phrases")
BRAND_NAMES[:10]

815 brand/designer phrases


['1 Piu 1 Uguale 3',
 '10 Crosby Derek Lam',
 '3.1 Phillip Lim',
 '6267',
 'A Degree Fahrenheit',
 'A Détacher',
 'A.F. Vandevorst',
 'A.L.C.',
 'A.P.C.',
 'A.W.A.K.E.']

# 2. Building a Fashion Ontology + SpaCy setup

In [6]:
# Cell 5: Download fashionpedia data 

if not ONTOLOGY_PATH.exists():
    url = "https://s3.amazonaws.com/ifashionist-dataset/annotations/instances_attributes_val2020.json"
    response = requests.get(url)

    with open("data/instances_attributes_val2020.json", "wb") as f:
        f.write(response.content)

In [7]:
# Cell 6: Fashionpedia supercategory to label function

def fp_supercategory_to_label(supercategory: str) -> str:
    label = supercategory.lower().strip()
    label = re.sub(r"[,/]+\s*", "_", label)
    label = re.sub(r"\s+", "_", label)
    label = re.sub(r"_+", "_", label).strip("_")
    return label

In [8]:
# Cell 7: Load Fashionpedia, assign category from supercategory directly

with open("data/instances_attributes_val2020.json") as f:
    data = json.load(f)

# Fashionpedia negative-label rows that should not become ontology terms
JUNK_TERMS = {
    "no non-textile material", "no special manufacturing technique",
    "no closures", "no opening",
}

rows = []

for cat in data["categories"]:
    supercategory = fp_supercategory_to_label(cat["supercategory"])
    for term in [t.strip().lower() for t in cat["name"].split(",")]:
        if term and term not in JUNK_TERMS:
            rows.append({"term": term, "category": supercategory})

for attr in data["attributes"]:
    term = re.sub(r"\s*\(.*?\)", "", attr["name"]).strip().lower()
    supercategory = fp_supercategory_to_label(attr["supercategory"])
    if term and term not in JUNK_TERMS:
        rows.append({"term": term, "category": supercategory})

fashionpedia_df = (
    pd.DataFrame(rows)
    .drop_duplicates(subset=["term"])
    .query("term != ''")
    # drop terms that still contain commas (multi-part attribute names e.g. "letters, numbers")
    .loc[lambda d: ~d["term"].str.contains(",")]
    .copy()
)

print("Fashionpedia categories (from supercategories):")
print(fashionpedia_df["category"].value_counts())
fashionpedia_df.head(5)

Fashionpedia categories (from supercategories):
category
nickname                                      122
silhouette                                     24
neckline_type                                  23
textile_pattern                                17
textile_finishing_manufacturing_techniques     16
length                                         14
decorations                                    10
upperbody                                       9
opening_type                                    9
non-textile_material_type                       9
garment_parts                                   7
waistline                                       7
head                                            5
legs_and_feet                                   5
animal                                          5
wholebody                                       4
others                                          4
lowerbody                                       3
leather                                    

,term,category
0,shirt,upperbody
1,blouse,upperbody
2,top,upperbody
3,t-shirt,upperbody
4,sweatshirt,upperbody


In [9]:
# Cell 8: Add fashionpedia ontology to data

fashionpedia_df.to_csv("data/fashionpedia_ontology.csv", index=False)

In [10]:
# Cell 9: Fashion term additions

# Colors: entirely absent from Fashionpedia
FASHION_COLORS = {
    "black", "white", "red", "blue", "green", "yellow", "pink", "gray",
    "grey", "gold", "silver", "beige", "cream", "ivory", "navy", "khaki",
    "brown", "orange", "purple", "burgundy", "camel", "nude", "ecru",
}

# Occasions/wear contexts
# not in Fashionpedia
FASHION_OCCASIONS = {
    "evening", "cocktail", "daywear", "resort", "bridal",
}

# Textile fibres (raw materials)
# from Wikipedia List of textile fibres.
# Banana and pineapple fibre omitted (not mentioned in fashion reviews).
FASHION_TEXTILE_FIBRES = {
    # animal
    "alpaca", "angora", "azlon", "byssus", "camel", "cashmere", "chiengora",
    "lambswool", "llama", "mohair", "qiviut", "rabbit", "silk", "eri",
    "spider silk", "vicuna", "wool", "yak",
    # plant / cellulosic
    "abaca", "bamboo", "kapok", "coir", "cotton", "flax", "hemp", "jute",
    "kenaf", "lyocell", "modal", "raffia", "ramie", "rayon", "sisal",
    # synthetic
    "nylon", "polyester", "spandex", "acrylic", "elastane", "kevlar",
    "lycra", "modacrylic", "nomex", "acetate",
}

# Fabric types
# from Wikipedia List of fabrics.
FASHION_FABRIC_TYPES = {
    # A
    "aertex", "baize", "barathea", "barkcloth", "batik", "batiste",
    "bedford cord", "bengaline silk", "bobbinet", "boiled wool", "bombazine",
    "boucle", "brilliantine", "broadcloth", "brocade", "broderie anglaise",
    "buckram", "burlap",
    # C
    "calico", "cambric", "camlet", "canvas", "challis", "charmeuse",
    "charvet", "cheesecloth", "chenille", "chiffon", "chino", "chintz",
    "cloque", "corduroy", "cotton duck", "crash", "crepe", "crepe de chine",
    "cretonne", "crochet",
    # D
    "damask", "denim", "dimity", "dobby", "donegal tweed", "dotted swiss",
    "double cloth", "drill", "drugget", "duck", "dupioni silk", "dungaree",
    # E–F
    "eolienne", "etamine", "faux fur", "faux leather", "felt", "fishnet",
    "flannel", "flannelette", "foulard", "fustian",
    # G
    "gabardine", "gauze", "gazar", "georgette", "ghalamkar", "gingham",
    "grenadine", "grosgrain",
    # H
    "habutai", "harris tweed", "herringbone",
    # I–J
    "interlock jersey", "jacquard knit", "jamdani", "jersey",
    # K–L
    "knit", "lace", "lame", "lampas", "lawn cloth", "loden",
    "longcloth", "linsey-woolsey",
    # M
    "mackinaw", "madras", "matelasse", "melton", "mesh", "moire",
    "moleskin", "moquette", "mousseline", "muslin",
    # N–O
    "neoprene", "net", "oilskin", "organdy", "organza", "ottoman",
    # P
    "peau de soie", "percale", "pique", "plisse", "plush", "polar fleece",
    "pongee", "poplin",
    # R
    "rep", "rib knit", "ripstop",
    # S
    "sailcloth", "sateen", "satin", "scrim", "seersucker", "serge",
    "shantung", "sharkskin", "shot silk", "stockinette", "suede", "surah",
    # T
    "taffeta", "tapestry", "tartan", "terrycloth", "toile", "tricot",
    "tulle", "tweed", "twill",
    # V
    "velour", "velvet", "velveteen", "voile",
    # W–Z
    "whipcord", "worsted wool", "zephyr",
}

# Terms not covered by Fashionpedia or the sets above.
# Format: "term": ("category", "subcategory")
FASHION_SUPPLEMENTARY = {
    "leather":    ("material",  "non_textile_material_type"),
    # garments absent from Fashionpedia
    "suit":       ("garment",   "wholebody"),
    "trouser":    ("garment",   "lowerbody"),
    # garment parts absent from Fashionpedia
    "shoulder":   ("garment",   "garment_part"),
    "bodice":     ("garment",   "garment_part"),
    "hem":        ("garment",   "garment_part"),
    "seam":       ("garment",   "garment_part"),
    "panel":      ("garment",   "garment_part"),
    "trim":       ("garment",   "garment_part"),
    "waist":      ("garment",   "garment_part"),
    # embellishment
    "embroidery": ("style",     "embellishment"),
    # closures
    "zip":        ("garment",   "closure"),
    "button":     ("garment",   "closure"),
    # style descriptors
    "shape":      ("style",     "style_descriptor"),
}

# Style/aesthetic descriptors
FASHION_STYLE_DESCRIPTORS = {
    "tailored", "structured", "sheer", "draped", "cropped", "oversized",
    "fitted", "fluid", "minimal", "romantic", "sporty", "luxurious", "ornate",
    "embellished", "textured", "sleek", "feminine", "masculine", "voluminous",
    "layered", "opulent", "graphic", "monochrome", "vintage", "sculptural",
    "elegant", "casual", "deconstructed", "slouchy", "dramatic", "polished",
    "long", "wide", "slim",
}

# Set to True to include brand/designer names in fashion tagging
COUNT_BRANDS_AS_FASHION = True

### SpaCy setup

In [11]:
# Cell 10: spaCy setup

try:
    nlp = spacy.load("en_core_web_sm", disable=["ner"])
except OSError:
    raise OSError(
        "spaCy model not installed. Run:\n  python -m spacy download en_core_web_sm"
    )

nlp.max_length = max(2_000_000, nlp.max_length)

In [12]:
# Cell 11: Build ontology dataframe

FP_SUPERCATEGORY_REMAP = {
    "upperbody":                                  ("garment",   "upperbody"),
    "lowerbody":                                  ("garment",   "lowerbody"),
    "wholebody":                                  ("garment",   "wholebody"),
    "waist":                                      ("accessory", None),
    "head":                                       ("accessory", "head"),
    "neck":                                       ("accessory", None),
    "arms_and_hands":                             ("accessory", "arms_and_hands"),
    "legs_and_feet":                              ("accessory", "legs_and_feet"),
    "animal":                                     ("style",     "textile_pattern"),
    "nickname":                                   ("garment",   "garment_type"),
    "garment_parts":                              ("garment",   "garment_part"),
    "silhouette":                                 ("style",     "silhouette"),
    "neckline_type":                              ("garment",   "neckline_type"),
    "textile_pattern":                            ("style",     "textile_pattern"),
    "textile_finishing_manufacturing_techniques": ("style",     "textile_technique"),
    "length":                                     ("style",     "length"),
    "decorations":                                ("style",     "embellishment"),
    "opening_type":                               ("garment",   "opening_type"),
    "non-textile_material_type":                  ("material",  "non_textile_material_type"),
    "leather":                                    ("material",  "non_textile_material_type"),
    "waistline":                                  ("garment",   "waistline"),
    "others":                                     ("accessory", None),
    "closures":                                   ("garment",   "closure"),
}

def make_rows(terms, category, subcategory=None):
    return [{"term": t.lower(), "category": category, "subcategory": subcategory} for t in terms]

pieces = [fashionpedia_df[["term", "category"]].assign(subcategory=None).copy()]
pieces.append(pd.DataFrame(make_rows(FASHION_COLORS,           "color")))
pieces.append(pd.DataFrame(make_rows(FASHION_OCCASIONS,        "occasion")))
pieces.append(pd.DataFrame(make_rows(FASHION_TEXTILE_FIBRES,   "material", "textile_fibre")))
pieces.append(pd.DataFrame(make_rows(FASHION_FABRIC_TYPES,     "material", "fabric_type")))
pieces.append(pd.DataFrame([
    {"term": t.lower(), "category": cat, "subcategory": sub}
    for t, (cat, sub) in FASHION_SUPPLEMENTARY.items()
]))
pieces.append(pd.DataFrame(make_rows(FASHION_STYLE_DESCRIPTORS, "style", "style_descriptor")))
if COUNT_BRANDS_AS_FASHION:
    pieces.append(pd.DataFrame(make_rows(BRAND_PHRASES, "brand")))

ontology_df = (
    pd.concat(pieces, ignore_index=True)
    .drop_duplicates(subset=["term"])
    .query("term != ''")
    .copy()
)

for fp_cat, (new_cat, new_sub) in FP_SUPERCATEGORY_REMAP.items():
    mask = ontology_df["category"] == fp_cat
    ontology_df.loc[mask, "category"]    = new_cat
    ontology_df.loc[mask, "subcategory"] = new_sub

ontology_df["n_words"] = ontology_df["term"].str.split().str.len()

print(f"Ontology before lemmatisation: {len(ontology_df)} terms")
print("\nBy category:")
print(ontology_df["category"].value_counts())
print("\nBy subcategory (non-null):")
print(ontology_df[ontology_df["subcategory"].notna()]["subcategory"].value_counts())

Ontology before lemmatisation: 1366 terms

By category:
category
brand        813
garment      196
material     191
style        121
color         22
accessory     18
occasion       5
Name: count, dtype: int64

By subcategory (non-null):
subcategory
fabric_type                  136
garment_type                 122
textile_fibre                 42
style_descriptor              34
silhouette                    24
neckline_type                 23
textile_pattern               22
textile_technique             16
length                        14
garment_part                  13
non_textile_material_type     13
embellishment                 11
upperbody                      9
opening_type                   9
waistline                      7
wholebody                      5
head                           5
legs_and_feet                  5
lowerbody                      4
closure                        4
arms_and_hands                 2
Name: count, dtype: int64


In [13]:
# Cell 12: Lemmatise ontology terms, drop duplicates, recompute n_words

def lemmatise_term(term: str) -> str:
    doc = nlp(term)
    return " ".join(t.lemma_.lower() for t in doc)

mask_non_brand = ontology_df["category"] != "brand"
ontology_df.loc[mask_non_brand, "term"] = ontology_df.loc[mask_non_brand, "term"].apply(lemmatise_term)

mask_comma = ontology_df["term"].str.contains(",") & mask_non_brand
ontology_df = ontology_df[~mask_comma].drop_duplicates(subset=["term"]).copy()
ontology_df["n_words"] = ontology_df["term"].str.split().str.len()

In [14]:
# Cell 13: Derive lookup sets from ontology dataframe

ONTOLOGY_UNIGRAMS = set(ontology_df.loc[ontology_df["n_words"] == 1, "term"])
ONTOLOGY_BIGRAMS  = set(ontology_df.loc[ontology_df["n_words"] == 2, "term"])
ONTOLOGY_TRIGRAMS = set(ontology_df.loc[ontology_df["n_words"] == 3, "term"])
TERM_TO_CATEGORY  = dict(zip(ontology_df["term"], ontology_df["category"]))

print(f"Ontology: {len(ONTOLOGY_UNIGRAMS)} unigrams, {len(ONTOLOGY_BIGRAMS)} bigrams, {len(ONTOLOGY_TRIGRAMS)} trigrams")
ontology_df.sample(10)

Ontology: 719 unigrams, 491 bigrams, 106 trigrams


,term,category,subcategory,n_words
659,zadig & voltaire,brand,None,3
343,lycra,material,textile_fibre,1
631,daks by giles deacon,brand,None,4
37,epaulette,garment,garment_part,1
814,misha nonoo,brand,None,2
685,han ahn soon,brand,None,3
449,peau de soie,material,fabric_type,3
1311,wc,brand,None,1
85,bolero,garment,garment_type,1
673,a degree fahrenheit,brand,None,3


In [15]:
# Cell 14: Save ontology dataframe to CSV

ontology_df[["term", "category", "subcategory", "n_words"]].fillna("none").to_csv(
    "data/fashion_ontology.csv", index=False
)

# 3. POS tagging

In [16]:
# Cell 15: Load data
df = pd.read_csv(DATA_PATH).reset_index(names="doc_id")


df["_date"] = pd.to_datetime(df[DATE_COL], format="%d-%b-%y")

print(df.shape)
print(f"Date range: {df['_date'].min().date()} → {df['_date'].max().date()}")

(6629, 9)
Date range: 1999-09-12 → 2014-06-12


In [17]:
# Cell 16: Text normalisation

CONTRACTION_MAP = {
    r"\bcan['\'']t\b": "cannot",
    r"\bwon['\'']t\b": "will not",
    r"n['\'']t\b":       " not",
    r"['\'']re\b":       " are",
    r"['\'']ve\b":       " have",
    r"['\'']ll\b":       " will",
    r"['\'']d\b":        " would",
    r"['\'']m\b":        " am",
    r"['\'']s\b":        " s",
}

punctuation_to_space = str.maketrans({p: " " for p in string.punctuation})

def normalize_text(text: str) -> str:
    text = str(text)
    for pattern, repl in CONTRACTION_MAP.items():
        text = re.sub(pattern, repl, text, flags=re.IGNORECASE)
    text = text.translate(punctuation_to_space)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["review_norm"] = df[TEXT_COL].fillna("").apply(normalize_text)
df[[TEXT_COL, "review_norm"]].head(2)

,review,review_norm
0,Designer Matt Nye's sophomore show featured a ...,Designer Matt Nye s sophomore show featured a ...
1,"Armani proposed a light, feminine silhouette f...",Armani proposed a light feminine silhouette fo...


In [18]:
# Cell 17: Brand mention extraction (preserves brand info separately from model text)

def extract_brand_mentions(text: str, brand_phrases=BRAND_PHRASES):
    text_l = str(text).lower()
    return sorted(
        [b for b in brand_phrases if re.search(rf"\b{re.escape(b)}\b", text_l)],
        key=len, reverse=True
    )

df["brand_mentions"] = df[TEXT_COL].fillna("").apply(extract_brand_mentions)
df[[BRAND_COL, "brand_mentions"]].head(10)

,designer,brand_mentions
0,Matt Nye,"[sophomore, matt nye]"
1,Giorgio Armani,[sea]
2,Eric Bergère,[]
3,Céline,"[michael kors, trademark]"
4,Byblos,"[john bartlett, byblos]"
5,Bottega Veneta,[bottega veneta]
6,Thimister,[thimister]
7,Rebecca Danenberg,"[rebecca danenberg, back]"
8,Jill Stuart,[jill stuart]
9,Loewe,"[narciso rodriguez, loewe]"


In [19]:
# Cell 18: POS tagging on normalised text

def spacy_process(texts, batch_size=64):
    rows = []
    for doc_id, doc in zip(df["doc_id"], nlp.pipe(texts, batch_size=batch_size)):
        for token in doc:
            if token.is_space:
                continue
            rows.append({
                "doc_id":        doc_id,
                "token":         token.text,
                "lemma":         token.lemma_.lower().strip(),
                "pos":           token.pos_,
                "tag":           token.tag_,
                "is_alpha":      token.is_alpha,
                "is_stop":       token.is_stop,
                "is_brand_word": token.text.lower() in BRAND_WORDS,
            })
    return pd.DataFrame(rows)
token_df = spacy_process(df["review_norm"].tolist())


In [20]:
token_df.sample(5)

,doc_id,token,lemma,pos,tag,is_alpha,is_stop,is_brand_word
488721,2318,pair,pair,NOUN,NN,True,False,False
849434,3825,were,be,VERB,VBD,True,True,False
1066347,4651,dress,dress,VERB,VB,True,False,True
1342621,5784,approach,approach,NOUN,NN,True,False,False
1475365,6272,insanity,insanity,NOUN,NN,True,False,False


In [21]:
token_df.to_pickle(OUT_DIR / "df_noun_tokens_unfiltered.pkl")

## 3.1 Filter out stopwords

In [ ]:
# Cell 19: Domain stop nouns definitions

DOMAIN_STOP_NOUNS = {
    # Words for brand
    "brand", "label", "house", "designer",
    # Vague words
    "idea", "thing", "way", "kind", "version", "sort", "piece",
    "mix", "sense", "touch", "world", "new", "view", "theme", "trend",
    "element", "detail", "feature", "aspect", "factor", "point", "quality", "effect",
    "concept", "inspiration", "influence", "mood", "vibe", "ambience", "aesthetic", 
    "atmosphere", "impression",
    # Uninformative runway words
    "collection", "season", "show", "look", "fashion", "clothe",
    "color", "fabric", "runway", "model", "style", "top", "bottom",
    "silhouette", "shape", "length", "pattern", "embellishment",
    "dress", "garment", "clothing", "outfit", "apparel", "wear",
    # Size indicators
    "little", "long", "wide", "slim", "lot", "plenty", "bit",
    # Person indicators
    "woman", "girl", "man",
    # Time indicators (not relevant for topics)
    "time", "day", "week", "month", "year",
    "decade", "century", "era", "period",
    "moment", "age", "duration",
    "today", "tomorrow", "yesterday",
    "present", "past", "future",
    "beginning", "start", "end", "middle",
    "aftermath", "afternoon", "morning",
    "evening", "night", "occasion", "instance",
    # Time adjectives
    "new", "old", "recent", "current",
    "former", "previous", "early", "late",
    "earlier", "later", "upcoming",
    "first", "second", "third", "final",
    "initial", "last", "next"
}


In [23]:
# Cell 20a: Filter tokens, keep non-stop nouns, drop domain-generic nouns
VALID_POS_FOR_MODELLING = {"NOUN", "ADJ"}

token_df = pd.read_pickle(OUT_DIR / "df_noun_tokens_unfiltered.pkl")
token_df = token_df[
    token_df["is_alpha"] &
    token_df["lemma"].ne("") &
    token_df["pos"].isin(VALID_POS_FOR_MODELLING) &
    (~token_df["is_stop"]) &
    (token_df["lemma"].str.len() >= 3) &
    (~token_df["lemma"].isin(DOMAIN_STOP_NOUNS))
].copy()

token_df.head()

,doc_id,token,lemma,pos,tag,is_alpha,is_stop,is_brand_word
4,0,sophomore,sophomore,NOUN,NN,True,False,True
8,0,coed,coed,NOUN,NN,True,False,False
13,0,sailor,sailor,NOUN,NN,True,False,False
14,0,pants,pant,NOUN,NNS,True,False,False
16,0,jackets,jacket,NOUN,NNS,True,False,False


In [24]:
# Cell 20b: Build per-document noun token lists

doc_nouns = (
    token_df.groupby("doc_id")["lemma"]
    .apply(list)
    .rename("noun_tokens")
    .reset_index()
)

df = df.merge(doc_nouns, on="doc_id", how="left")
df["noun_tokens"] = df["noun_tokens"].apply(lambda x: x if isinstance(x, list) else [])
df["noun_text"]   = df["noun_tokens"].apply(lambda toks: " ".join(toks))

df[["doc_id", "noun_tokens", "noun_text"]].head(3)

,doc_id,noun_tokens,noun_text
0,0,"[sophomore, coed, sailor, pant, jacket, simple...",sophomore coed sailor pant jacket simple cotto...
1,1,"[light, feminine, millennium, line, foam, fuch...",light feminine millennium line foam fuchsia li...
2,2,"[tailleur, skirt, pleat, sweater, ruched, shir...",tailleur skirt pleat sweater ruched shirt inte...


In [ ]:
# Cell 20c: Compute term-level stats for Stage 1 diagnostics

term_doc_counts = (
    token_df.groupby(["lemma", "doc_id"])
    .size()
    .rename("count")
    .reset_index()
)

N_DOCS = token_df["doc_id"].nunique()


doc_freq = (
    term_doc_counts.groupby("lemma")["doc_id"]
    .nunique()
    .rename("doc_freq")
)


def _doc_entropy(counts: np.ndarray, n_docs: int) -> float:
    """Normalised Shannon entropy of term occurrences across documents."""
    if len(counts) <= 1:
        return 0.0
    probs = counts / counts.sum()
    raw   = scipy_entropy(probs, base=2)          # bits
    max_e = np.log2(n_docs)                       # theoretical max
    return float(raw / max_e) if max_e > 0 else 0.0

doc_entropy = (
    term_doc_counts
    .groupby("lemma")["count"]
    .apply(lambda s: _doc_entropy(s.values, N_DOCS))
    .rename("doc_entropy")
)


year_lookup = df[["doc_id", "_date"]].copy()
year_lookup["year"] = year_lookup["_date"].dt.year

term_year = (
    token_df[["lemma", "doc_id"]]
    .drop_duplicates()
    .merge(year_lookup[["doc_id", "year"]], on="doc_id", how="left")
)

temporal_coverage = (
    term_year.groupby("lemma")["year"]
    .nunique()
    .rename("temporal_coverage")
)

years_present = (
    term_year.groupby("lemma")["year"]
    .apply(lambda s: sorted(s.dropna().unique().tolist()))
    .rename("years_active")
)

# Assemble df_with_term_stats
term_stats_df = (
    pd.concat([doc_freq, doc_entropy, temporal_coverage, years_present], axis=1)
    .reset_index()
    .rename(columns={"index": "lemma"})
)

term_stats_df["doc_freq_prop"] = term_stats_df["doc_freq"] / N_DOCS

print(f"{len(term_stats_df):,} unique terms")
print(term_stats_df.sort_values("doc_entropy", ascending=False).head(10))

18,817 unique terms
         lemma  doc_freq  doc_entropy  temporal_coverage  \
8576    jacket      2948     0.897266                 16   
14828    skirt      2846     0.893886                 16   
1515     black      2803     0.888322                 16   
2976      coat      2305     0.869859                 16   
12542    print      2248     0.864853                 16   
9041   leather      2076     0.854559                 16   
11498     pant      1834     0.847253                 16   
18442    white      1902     0.847195                 16   
14692     silk      1690     0.835866                 16   
6958      good      1633     0.833129                 16   

                                            years_active  doc_freq_prop  
8576   [1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...       0.444981  
14828  [1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...       0.429585  
1515   [1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...       0.423094  
2976   [1999, 2000, 200

In [ ]:
# Cell 20d: Diagnostic plots for entropy and temporal coverage
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Document entropy distribution
axes[0].hist(term_stats_df["doc_entropy"], bins=60, edgecolor="none")
p95 = term_stats_df["doc_entropy"].quantile(0.95)
axes[0].axvline(p95, color="red", linestyle="--", label=f"95th pct = {p95:.3f}")
axes[0].set_title("Document entropy (normalised)")
axes[0].set_xlabel("Entropy")
axes[0].set_ylabel("# terms")
axes[0].legend()

# Temporal coverage distribution
axes[1].hist(term_stats_df["temporal_coverage"], bins=range(1, term_stats_df["temporal_coverage"].max() + 2),
             edgecolor="none", align="left")
axes[1].set_title("Temporal coverage (# years term appears)")
axes[1].set_xlabel("Years")
axes[1].set_ylabel("# terms")

# Entropy vs temporal coverage
axes[2].scatter(
    term_stats_df["temporal_coverage"],
    term_stats_df["doc_entropy"],
    s=4, alpha=0.3
)
axes[2].axhline(p95, color="red", linestyle="--", label=f"95th pct entropy")
axes[2].set_title("Entropy vs temporal coverage")
axes[2].set_xlabel("Temporal coverage (years)")
axes[2].set_ylabel("Doc entropy (normalised)")
axes[2].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / "term_stats_diagnostics.png", dpi=150)
plt.show()


top_entropy = (
    term_stats_df[term_stats_df["doc_freq"] > 1]
    .nlargest(20, "doc_entropy")[["lemma", "doc_freq", "doc_entropy", "temporal_coverage"]]
)
print("\nHighest-entropy surviving terms")
print(top_entropy.to_string(index=False))


Highest-entropy surviving terms
    lemma  doc_freq  doc_entropy  temporal_coverage
   jacket      2948     0.897266                 16
    skirt      2846     0.893886                 16
    black      2803     0.888322                 16
     coat      2305     0.869859                 16
    print      2248     0.864853                 16
  leather      2076     0.854559                 16
     pant      1834     0.847253                 16
    white      1902     0.847195                 16
     silk      1690     0.835866                 16
     good      1633     0.833129                 16
    short      1465     0.820219                 16
     line      1436     0.818635                 16
     high      1253     0.805095                 16
    shirt      1176     0.796298                 16
     suit      1125     0.792577                 16
     gown      1143     0.792268                 16
 shoulder      1105     0.791108                 16
  trouser      1061     0.78812

/var/folders/_p/f1mg9b290kx6bf1ytjkd60qm0000gn/T/ipykernel_37126/977667682.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# 4. Checkpoint: save output

In [ ]:
# Cell 21: Save preprocessed df with noun tokens and term stats

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

df.to_pickle(CHECKPOINT_DIR / "df_with_noun_tokens.pkl")
print(f"Saved: {CHECKPOINT_DIR / 'df_with_noun_tokens.pkl'}  ({len(df):,} rows)")

term_stats_df.to_pickle(CHECKPOINT_DIR / "df_with_term_stats.pkl")
print(f"Saved: {CHECKPOINT_DIR / 'df_with_term_stats.pkl'}  ({len(term_stats_df):,} terms)")

Saved: checkpoints/df_with_noun_tokens.pkl  (6,629 rows)
Saved: checkpoints/df_with_term_stats.pkl  (18,817 terms)
